# Lecture 28: Practice Problems Master Solutions Notebook
## Supervised Machine Learning (Part 4) — Decision Trees

This notebook provides verified production solutions and unit tests for the 4 Case Studies and 8 Algorithmic Challenges.

### Case Study 1: E-Commerce Session Conversion & Pruning (ShopSmart Dataset)
Pipeline implementation using ColumnTransformer, DecisionTreeClassifier, GridSearchCV, and cost-complexity pruning path.

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import classification_report, f1_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

csv_path = r'c:\Users\Shivam Keshari\OneDrive\Desktop\AI-ML\lecture_28\shop_smart_ecommerce.csv'
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    X = df.drop('Revenue', axis=1)
    y = df['Revenue'].astype(int)

    cat_cols = ['Month', 'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'VisitorType', 'Weekend']
    num_cols = [c for c in X.columns if c not in cat_cols]

    preprocessor = ColumnTransformer([
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ])

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    X_train_proc = preprocessor.fit_transform(X_train)
    X_test_proc = preprocessor.transform(X_test)

    # Unconstrained tree
    unconstrained = DecisionTreeClassifier(random_state=42)
    unconstrained.fit(X_train_proc, y_train)
    print('Unconstrained Depth:', unconstrained.get_depth(), 'Leaves:', unconstrained.get_n_leaves())
    print('Unconstrained Test F1:', round(f1_score(y_test, unconstrained.predict(X_test_proc)), 4))

    # Pruned tree with max_depth=5
    pruned = DecisionTreeClassifier(max_depth=5, min_samples_leaf=20, random_state=42)
    pruned.fit(X_train_proc, y_train)
    f1_pruned = f1_score(y_test, pruned.predict(X_test_proc))
    print('Pruned Model Test F1:', round(f1_pruned, 4))
    assert f1_pruned > 0.55, 'Model must beat F1 > 0.55 benchmark'
    print('Case Study 1 Passed!')


Unconstrained Depth: 25 Leaves: 803
Unconstrained Test F1: 0.5497
Pruned Model Test F1: 0.6457
Case Study 1 Passed!


### Case Study 2: Hospital Emergency Triage & High-Cardinality ID Failure
Pure NumPy ShannonEntropyCalculator and GainRatioOptimizer.

In [2]:
class ShannonEntropyCalculator:
    @staticmethod
    def compute(labels):
        labels = np.asarray(labels)
        if len(labels) == 0:
            return 0.0
        _, counts = np.unique(labels, return_counts=True)
        probs = counts / counts.sum()
        non_zero_probs = probs[probs > 0]
        return float(-np.sum(non_zero_probs * np.log2(non_zero_probs)))

class GainRatioOptimizer:
    @staticmethod
    def compute(parent_labels, child_subsets):
        parent_entropy = ShannonEntropyCalculator.compute(parent_labels)
        total_n = len(parent_labels)
        weighted_child_entropy = 0.0
        split_info = 0.0
        for subset in child_subsets:
            sub_n = len(subset)
            if sub_n == 0:
                continue
            w = sub_n / total_n
            weighted_child_entropy += w * ShannonEntropyCalculator.compute(subset)
            split_info -= w * np.log2(w)
        ig = parent_entropy - weighted_child_entropy
        gain_ratio = ig / (split_info + 1e-12)
        return float(ig), float(split_info), float(gain_ratio)

# Unit Tests
entropy_calc = ShannonEntropyCalculator()
assert entropy_calc.compute([1, 1, 1]) == 0.0
assert round(entropy_calc.compute([0, 1]), 4) == 1.0

# Triage data
y_triage = np.array([1]*60 + [0]*40)
assert round(entropy_calc.compute(y_triage), 4) == 0.9710

# Patient ID (100 distinct subsets of size 1)
subsets_id = [[y_triage[i]] for i in range(100)]
ig_id, split_id, gr_id = GainRatioOptimizer.compute(y_triage, subsets_id)
assert round(ig_id, 4) == 0.9710
assert round(split_id, 4) == round(np.log2(100), 4)
assert gr_id < 0.20

# Heart Rate (50 high, 50 normal)
subsets_hr = [[1]*45 + [0]*5, [1]*15 + [0]*35]
ig_hr, split_hr, gr_hr = GainRatioOptimizer.compute(y_triage, subsets_hr)
assert gr_hr > gr_id, 'Gain ratio must reject Patient_ID in favor of Heart Rate'
print('Case Study 2 Passed!')


Case Study 2 Passed!


### Case Study 3: Municipal Property Valuation (Regression Trees & Variance Reduction)
Pure NumPy ContinuousSplitFinder implementing midpoint thresholding and variance drop.

In [3]:
class ContinuousSplitFinder:
    @staticmethod
    def find_best_split(x, y):
        x = np.asarray(x)
        y = np.asarray(y)
        sorted_idx = np.argsort(x)
        x_sort = x[sorted_idx]
        y_sort = y[sorted_idx]
        
        parent_var = np.var(y)
        n = len(y)
        
        unique_x = np.unique(x_sort)
        if len(unique_x) < 2:
            return None, 0.0
            
        candidate_thresholds = (unique_x[:-1] + unique_x[1:]) / 2.0
        best_thresh = None
        best_vr = -1.0
        
        for thresh in candidate_thresholds:
            mask = x <= thresh
            y_l = y[mask]
            y_r = y[~mask]
            if len(y_l) == 0 or len(y_r) == 0:
                continue
            weighted_child_var = (len(y_l)/n) * np.var(y_l) + (len(y_r)/n) * np.var(y_r)
            vr = parent_var - weighted_child_var
            if vr > best_vr:
                best_vr = vr
                best_thresh = thresh
        return best_thresh, best_vr

# Property valuation test
area = np.array([600, 800, 950, 1100, 1300, 1500, 1800, 2200])
rent = np.array([18.0, 22.0, 25.0, 28.0, 36.0, 40.0, 48.0, 55.0])
best_t, best_v = ContinuousSplitFinder.find_best_split(area, rent)
assert best_t == 1200.0, 'Best split threshold must be 1200 sq ft'
assert round(best_v, 2) == 115.56, 'Variance reduction must be 115.56'
print('Case Study 3 Passed!')


Case Study 3 Passed!


### Case Study 4: Mathematical Equivalence (Gini Impurity Evaluator)
Pure NumPy Gini Impurity calculator and verification against theoretical bounds.

In [4]:
class GiniImpurityEvaluator:
    @staticmethod
    def compute(labels):
        labels = np.asarray(labels)
        if len(labels) == 0:
            return 0.0
        _, counts = np.unique(labels, return_counts=True)
        probs = counts / counts.sum()
        return float(1.0 - np.sum(probs ** 2))

gini_calc = GiniImpurityEvaluator()
assert gini_calc.compute([1, 1, 1]) == 0.0
assert gini_calc.compute([0, 1]) == 0.5
assert gini_calc.compute([0, 1, 2, 3]) == 0.75
print('Case Study 4 Passed!')
print('All Lecture 28 Master Solutions Verified!')


Case Study 4 Passed!
All Lecture 28 Master Solutions Verified!
